# PIMA Diabetes Classification with Optuna + XGBoost

This notebook builds a binary classifier for diabetes prediction using the PIMA Indians dataset and tunes XGBoost hyperparameters with Optuna.

## What this notebook covers
- Data loading and preprocessing
- Train/validation/test split (60/20/20)
- Optuna-driven hyperparameter optimization with pruning
- Final model training and ROC-AUC evaluation

## Goal
Maximize validation ROC-AUC while keeping generalization on the test set as stable as possible.

In [1]:
import optuna
import xgboost as xgb
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [ ]:
# Load dataset

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
# data cleaning
cols_with_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_zero] = df[cols_with_zero].replace(0, np.nan)

df.fillna(df.median(), inplace=True)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31,0
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.0,180.0,32.9,0.171,63,0
764,2,122.0,70.0,27.0,125.0,36.8,0.340,27,0
765,5,121.0,72.0,23.0,112.0,26.2,0.245,30,0
766,1,126.0,60.0,29.0,125.0,30.1,0.349,47,1


In [ ]:
# Split data (60:20:20)

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

In [6]:
def objective(trial):

    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "verbosity": 0,

        # Regularization
        "lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-8, 10.0, log=True),

        # Tree structure
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),

        # Sampling
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),

        # Learning
        "eta": trial.suggest_float("eta", 0.01, 0.3),
    }

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)

    pruning_callback = optuna.integration.XGBoostPruningCallback(
        trial, "validation-logloss"
    )

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=300,
        evals=[(dvalid, "validation")],
        early_stopping_rounds=30,
        callbacks=[pruning_callback],
    )

    preds = model.predict(dvalid)
    return roc_auc_score(y_valid, preds)

In [7]:
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.SuccessiveHalvingPruner()
)

study.optimize(objective, n_trials=50)


[I 2026-04-25 02:54:56,094] A new study created in memory with name: no-name-067c5b3b-6916-4437-9359-535a47fb7068


[0]	validation-logloss:0.64332
[1]	validation-logloss:0.63508
[2]	validation-logloss:0.62728
[3]	validation-logloss:0.62038
[4]	validation-logloss:0.61607
[5]	validation-logloss:0.61011
[6]	validation-logloss:0.60640
[7]	validation-logloss:0.59990
[8]	validation-logloss:0.59365
[9]	validation-logloss:0.58822
[10]	validation-logloss:0.58456
[11]	validation-logloss:0.57946
[12]	validation-logloss:0.57762
[13]	validation-logloss:0.57302
[14]	validation-logloss:0.57138
[15]	validation-logloss:0.56923
[16]	validation-logloss:0.56466
[17]	validation-logloss:0.56206
[18]	validation-logloss:0.55937
[19]	validation-logloss:0.55515
[20]	validation-logloss:0.55338
[21]	validation-logloss:0.55007
[22]	validation-logloss:0.54578
[23]	validation-logloss:0.54194
[24]	validation-logloss:0.53767
[25]	validation-logloss:0.53612
[26]	validation-logloss:0.53393
[27]	validation-logloss:0.53272
[28]	validation-logloss:0.52921
[29]	validation-logloss:0.52777
[30]	validation-logloss:0.52550
[31]	validation-lo

[I 2026-04-25 02:54:56,528] Trial 0 finished with value: 0.8694444444444444 and parameters: {'lambda': 0.06166849926934415, 'alpha': 2.0426727582736115, 'max_depth': 3, 'min_child_weight': 2, 'gamma': 0.3398611605047213, 'subsample': 0.9749975371934416, 'colsample_bytree': 0.664727661100575, 'eta': 0.02674045739391727}. Best is trial 0 with value: 0.8694444444444444.


[0]	validation-logloss:0.61528
[1]	validation-logloss:0.54478
[2]	validation-logloss:0.50917
[3]	validation-logloss:0.48718
[4]	validation-logloss:0.47404
[5]	validation-logloss:0.46759
[6]	validation-logloss:0.46302
[7]	validation-logloss:0.45242
[8]	validation-logloss:0.45273
[9]	validation-logloss:0.45103
[10]	validation-logloss:0.44749
[11]	validation-logloss:0.45579
[12]	validation-logloss:0.46062
[13]	validation-logloss:0.46520
[14]	validation-logloss:0.46978
[15]	validation-logloss:0.47877
[16]	validation-logloss:0.48186
[17]	validation-logloss:0.48495
[18]	validation-logloss:0.48792
[19]	validation-logloss:0.49103
[20]	validation-logloss:0.49330
[21]	validation-logloss:0.49434
[22]	validation-logloss:0.49425
[23]	validation-logloss:0.49854
[24]	validation-logloss:0.49334
[25]	validation-logloss:0.49681
[26]	validation-logloss:0.49935
[27]	validation-logloss:0.49884
[28]	validation-logloss:0.50060
[29]	validation-logloss:0.50060
[30]	validation-logloss:0.49878
[31]	validation-lo

[I 2026-04-25 02:54:56,632] Trial 1 finished with value: 0.8185185185185185 and parameters: {'lambda': 1.8196214548446498e-06, 'alpha': 0.0008474436793876865, 'max_depth': 9, 'min_child_weight': 5, 'gamma': 0.004173612661299177, 'subsample': 0.9946833679417597, 'colsample_bytree': 0.6336929595337979, 'eta': 0.2810539370282118}. Best is trial 0 with value: 0.8694444444444444.


...

[I 2026-04-25 02:54:59,668] Trial 47 pruned. Trial was pruned at iteration 2.


[0]	validation-logloss:0.61395
[1]	validation-logloss:0.56724


[I 2026-04-25 02:54:59,698] Trial 48 pruned. Trial was pruned at iteration 2.


[0]	validation-logloss:0.63081
[1]	validation-logloss:0.60391


[I 2026-04-25 02:54:59,728] Trial 49 pruned. Trial was pruned at iteration 2.


In [8]:
best_params = study.best_params

final_params = {
    **best_params,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "verbosity": 0,
}

In [9]:
dtrain_full = xgb.DMatrix(X_temp, label=y_temp)
dtest = xgb.DMatrix(X_test, label=y_test)

final_model = xgb.train(
    final_params,
    dtrain_full,
    num_boost_round=300
)

In [10]:
y_proba = final_model.predict(dtest)

print("Best Params:", study.best_params)
print("Best Validation AUC:", study.best_value)
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))

Best Params: {'lambda': 4.321228963536233e-07, 'alpha': 0.0002237742348738362, 'max_depth': 3, 'min_child_weight': 4, 'gamma': 1.1195756351090675e-08, 'subsample': 0.561556149437213, 'colsample_bytree': 0.5676139159388394, 'eta': 0.013915542743618298}
Best Validation AUC: 0.8696296296296295
Test ROC AUC: 0.8207407407407407


> The model shows mild overfitting: the validation ROC-AUC is about 0.05 higher than the test ROC-AUC.

## How to Improve Generalization

1. Use cross-validation inside Optuna (recommended)
Use `StratifiedKFold` in the objective so each trial is scored across folds.

```python
from sklearn.model_selection import StratifiedKFold
```

This reduces variance and gives a more stable AUC estimate.

2. Tune `scale_pos_weight`
The dataset is imbalanced, so include class weighting in the search space:

```python
"scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 5)
```

A practical baseline is:

```python
scale_pos_weight = (number_of_negative_class) / (number_of_positive_class)
```

3. Increase the number of Optuna trials
Try `n_trials=100` (or higher) for a better search over the parameter space.

4. Compare with a strong baseline
Benchmark against a Random Forest model to confirm XGBoost is meaningfully better.